# Beyond Acuity Prediction: A Triage Support Notebook
This notebook is designed to run end to end in Kaggle. It predicts `triage_acuity`, reports clinically relevant
validation metrics, highlights potential undertriage, and exports `submission.csv`.



## 1. Imports And Paths



In [ ]:
from __future__ import annotations

from pathlib import Path
from zipfile import ZipFile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

DATA_DIR = Path("/kaggle/input/triagegeist")
LOCAL_ARCHIVE = Path("../data/triagegeist.zip")
ID_COLUMN = "patient_id"
TARGET_COLUMN = "triage_acuity"
TEXT_COLUMN = "chief_complaint_raw"
HIGH_RISK_THRESHOLD = 2
LEAKAGE_COLUMNS = [ID_COLUMN, TARGET_COLUMN, "disposition", "ed_los_hours"]
SEED = 42

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 120)




## 2. Data Loading
The notebook supports the Kaggle competition dataset path directly. For local preview, it also falls back to the
downloaded zip archive if available.



In [ ]:
def read_csv(filename: str) -> pd.DataFrame:
    kaggle_path = DATA_DIR / filename
    if kaggle_path.exists():
        return pd.read_csv(kaggle_path)
    if LOCAL_ARCHIVE.exists():
        with ZipFile(LOCAL_ARCHIVE) as archive:
            with archive.open(filename) as handle:
                return pd.read_csv(handle)
    raise FileNotFoundError(f"Could not find {filename} in Kaggle input or local archive.")


train = read_csv("train.csv")
test = read_csv("test.csv")
complaints = read_csv("chief_complaints.csv")
history = read_csv("patient_history.csv")

train = train.merge(complaints[[ID_COLUMN, TEXT_COLUMN]], on=ID_COLUMN, how="left").merge(history, on=ID_COLUMN, how="left")
test = test.merge(complaints[[ID_COLUMN, TEXT_COLUMN]], on=ID_COLUMN, how="left").merge(history, on=ID_COLUMN, how="left")

display(pd.DataFrame(
    {
        "table": ["train", "test", "complaints", "history"],
        "rows": [len(train), len(test), len(complaints), len(history)],
        "cols": [train.shape[1], test.shape[1], complaints.shape[1], history.shape[1]],
    }
))




## 3. Leakage And Missingness Checks
`disposition` and `ed_los_hours` are excluded from training because they are post-triage outcomes rather than
real-time triage inputs.



In [ ]:
display(
    pd.DataFrame(
        {
            "excluded_column": LEAKAGE_COLUMNS,
            "reason": [
                "identifier",
                "target",
                "post-triage outcome",
                "post-triage outcome",
            ],
        }
    )
)

display(train.isna().mean().sort_values(ascending=False).head(12).to_frame("train_missing_rate"))
display(test.isna().mean().sort_values(ascending=False).head(12).to_frame("test_missing_rate"))




## 4. Feature Engineering
We add a small set of clinically motivated rule features and keyword flags to complement the structured intake data.



In [ ]:
KEYWORD_PATTERNS = {
    "kw_chest_pain": r"chest pain|thoracic pain|crushing chest",
    "kw_stroke_neuro": r"stroke|seizure|thunderclap|loss of vision|weakness|aphasia",
    "kw_respiratory_distress": r"shortness of breath|asthma|hypoxia|wheeze|near-drowning",
    "kw_trauma": r"trauma|fracture|haemothorax|stab|wound|fall|injury",
    "kw_overdose_toxic": r"overdose|poison|toxic|substance",
    "kw_bleeding": r"bleed|haemorrhage|hemorrhage|melena|hematemesis",
    "kw_pregnancy": r"pregnan|ectopic|postpartum|miscarriage",
    "kw_infection_sepsis": r"sepsis|fever|necrotising|infection|cellulitis",
}


def add_engineered_features(frame: pd.DataFrame) -> pd.DataFrame:
    df = frame.copy()
    complaint = df[TEXT_COLUMN].fillna("").str.lower()

    df["pain_unrecorded"] = (df["pain_score"] == -1).astype(int)
    df["pain_score_clean"] = df["pain_score"].replace(-1, np.nan)

    df["flag_low_oxygen"] = (df["spo2"] < 92).astype(int)
    df["flag_fever"] = (df["temperature_c"] >= 38.0).astype(int)
    df["flag_tachycardia"] = (df["heart_rate"] >= 100).astype(int)
    df["flag_tachypnea"] = (df["respiratory_rate"] >= 22).astype(int)
    df["flag_hypotension"] = (df["systolic_bp"] < 90).astype(int)
    df["flag_gcs_abnormal"] = (df["gcs_total"] < 15).astype(int)
    df["flag_high_news2"] = (df["news2_score"] >= 5).astype(int)
    df["flag_high_shock_index"] = (df["shock_index"] >= 0.9).astype(int)

    cardio_cols = [
        "hx_hypertension",
        "hx_heart_failure",
        "hx_atrial_fibrillation",
        "hx_coronary_artery_disease",
        "hx_peripheral_vascular_disease",
        "hx_stroke_prior",
    ]
    respiratory_cols = ["hx_asthma", "hx_copd"]
    neuro_cols = ["hx_dementia", "hx_epilepsy", "hx_stroke_prior"]
    frailty_cols = ["hx_dementia", "hx_ckd", "hx_malignancy", "hx_immunosuppressed"]

    df["cardio_burden"] = df[cardio_cols].sum(axis=1)
    df["respiratory_burden"] = df[respiratory_cols].sum(axis=1)
    df["neuro_burden"] = df[neuro_cols].sum(axis=1)
    df["frailty_burden"] = df[frailty_cols].sum(axis=1)

    for feature_name, pattern in KEYWORD_PATTERNS.items():
        df[feature_name] = complaint.str.contains(pattern, regex=True).astype(int)

    return df


train_feat = add_engineered_features(train).reset_index(drop=True)
test_feat = add_engineered_features(test).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(7, 4))
train_feat[TARGET_COLUMN].value_counts().sort_index().plot(kind="bar", ax=ax)
ax.set_title("Target Distribution")
ax.set_xlabel("ESI acuity")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()




## 5. Model Setup
The notebook uses a strong structured tabular model plus a lightweight complaint-text model. Their probabilities are
blended into a simple ensemble.



In [ ]:
def drop_leakage(frame: pd.DataFrame) -> pd.DataFrame:
    columns = [column for column in LEAKAGE_COLUMNS if column in frame.columns]
    return frame.drop(columns=columns).copy()


def feature_groups(frame: pd.DataFrame) -> tuple[list[str], list[str]]:
    numeric_columns = []
    categorical_columns = []
    for column in frame.columns:
        if column == TEXT_COLUMN:
            continue
        if pd.api.types.is_numeric_dtype(frame[column]):
            numeric_columns.append(column)
        else:
            categorical_columns.append(column)
    return numeric_columns, categorical_columns


def build_structured_model(numeric_columns: list[str], categorical_columns: list[str]) -> Pipeline:
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", SimpleImputer(strategy="median"), numeric_columns),
            (
                "cat",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
                    ]
                ),
                categorical_columns,
            ),
        ],
        remainder="drop",
    )
    model = HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_depth=8,
        max_iter=300,
        min_samples_leaf=40,
        l2_regularization=1.0,
        random_state=SEED,
    )
    return Pipeline([("preprocessor", preprocessor), ("model", model)])


def build_text_model() -> Pipeline:
    return Pipeline(
        steps=[
            ("tfidf", TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=5, max_features=30000, sublinear_tf=True)),
            ("model", ComplementNB(alpha=0.3)),
        ]
    )


def high_risk_recall(y_true: pd.Series, y_pred: np.ndarray) -> float:
    mask = y_true <= HIGH_RISK_THRESHOLD
    return float(np.mean(y_pred[mask.to_numpy()] <= HIGH_RISK_THRESHOLD))


X = drop_leakage(train_feat)
y = train_feat[TARGET_COLUMN].copy()
numeric_columns, categorical_columns = feature_groups(X)
structured_model = build_structured_model(numeric_columns, categorical_columns)
text_model = build_text_model()




## 6. Cross-Validation Results



In [ ]:
cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=SEED)
classes = np.sort(y.unique())
oof_structured = np.zeros((len(X), len(classes)))
oof_text = np.zeros((len(X), len(classes)))

for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
    X_train_fold = X.iloc[train_idx]
    y_train_fold = y.iloc[train_idx]
    X_valid_fold = X.iloc[valid_idx]

    structured_fold = clone(structured_model)
    structured_fold.fit(X_train_fold, y_train_fold)
    oof_structured[valid_idx] = structured_fold.predict_proba(X_valid_fold)

    text_fold = clone(text_model)
    text_fold.fit(X_train_fold[TEXT_COLUMN], y_train_fold)
    oof_text[valid_idx] = text_fold.predict_proba(X_valid_fold[TEXT_COLUMN])
    print(f"completed fold {fold}")

oof_prob = 0.8 * oof_structured + 0.2 * oof_text
oof_pred = classes[np.argmax(oof_prob, axis=1)]

metrics = {
    "macro_f1": f1_score(y, oof_pred, average="macro"),
    "high_risk_recall": high_risk_recall(y, oof_pred),
    "undertriage_rate": float(np.mean((oof_pred - y.to_numpy()) >= 2)),
}
display(pd.DataFrame([metrics]))

cm = confusion_matrix(y, oof_pred, labels=classes, normalize="true")
display(pd.DataFrame(cm, index=classes, columns=classes).round(3))




## 7. Undertriage Review



In [ ]:
undertriage = train_feat[[ID_COLUMN, TEXT_COLUMN, "news2_score", "spo2", "gcs_total", "age_group", "language", "arrival_mode"]].copy()
undertriage["actual_acuity"] = y.to_numpy()
undertriage["predicted_acuity"] = oof_pred
undertriage["prediction_gap"] = undertriage["predicted_acuity"] - undertriage["actual_acuity"]
undertriage = undertriage[undertriage["prediction_gap"] >= 2].sort_values("prediction_gap", ascending=False)
display(undertriage.head(20))




## 8. Subgroup Audit



In [ ]:
rows = []
for column in ["age_group", "sex", "language", "site_id", "arrival_mode"]:
    for value, group in train_feat.groupby(column, dropna=False):
        if len(group) < 100:
            continue
        idx = group.index.to_numpy()
        rows.append(
            {
                "subgroup": column,
                "value": value,
                "count": len(group),
                "macro_f1": f1_score(y.iloc[idx], oof_pred[idx], average="macro"),
                "high_risk_recall": high_risk_recall(y.iloc[idx], oof_pred[idx]),
                "undertriage_rate": float(np.mean((oof_pred[idx] - y.iloc[idx].to_numpy()) >= 2)),
            }
        )
subgroup_audit = pd.DataFrame(rows).sort_values(["subgroup", "macro_f1"], ascending=[True, False])
display(subgroup_audit.head(30))




## 9. Fit Final Models And Export Submission



In [ ]:
structured_model.fit(X, y)
text_model.fit(X[TEXT_COLUMN], y)

X_test = drop_leakage(test_feat)
structured_test_prob = structured_model.predict_proba(X_test)
text_test_prob = text_model.predict_proba(X_test[TEXT_COLUMN])
test_prob = 0.8 * structured_test_prob + 0.2 * text_test_prob
test_pred = classes[np.argmax(test_prob, axis=1)]

submission = pd.DataFrame({ID_COLUMN: test_feat[ID_COLUMN], TARGET_COLUMN: test_pred})
submission.to_csv("submission.csv", index=False)
display(submission.head())




## 10. Limitations
- The data is synthetic and should not be treated as evidence of real-world clinical deployment performance.
- The model is intended as decision support, not a replacement for triage staff.
- Subgroup findings here are useful for stress-testing but not strong fairness claims about real hospitals.
## 11. Reproducibility
This notebook runs end to end, creates a submission file, and keeps the leakage constraints explicit.
